# Building and extending phlex-examples, with git and CMake

This notebook walks through the [phlex-examples](https://github.com/Framework-R-D/phlex-examples)
repository. Along the way it covers the git and CMake you need to work on Phlex code.

When you finish, you will be able to:
- clone a repository, pick the branch that matches your Phlex version, and make your own branch
- explain what CMake does, and read the `CMakeLists.txt` of a Phlex project
- build phlex-examples and run its C++, unfold and Python workflows
- add a new Phlex algorithm, register it with CMake, and run it
- use git to see what you changed, undo a mistake, and commit your work

Shell commands run in code cells that start with `!` or `%%bash`. When a step asks you to
**edit a file**, open it from the JupyterLab file browser on the left, make the change, and save it
(Ctrl+S) before you run the next cell. Most edits also have a **solution cell** that makes the change
for you, so you can catch up if you fall behind.

# Section 0: Software setup

You need **phlex**, **CMake 3.31 or later**, a C++ compiler and **git**. On the DUNE Tech JupyterHub
these come from the phaindia apptainer image.

**Choose the phaindia apptainer kernel** from the kernel menu at the top right of this notebook
(or **Kernel → Change Kernel…**). The kernel runs inside the image, so every `!` and `%%bash` cell
below can see phlex.

Run the next cell to check that the tools are visible. Run it again if you restart the kernel.

In [ ]:
!which phlex cmake git c++
!cmake --version | head -1
!git --version
!phlex --version

If `git` is missing from the image, open a terminal **outside** the container (File → New → Terminal)
and run the git commands there. The files are the same in both places.

## Paths

You will work in `~/phlex-work`, not in the directory this notebook came from. The hub keeps this
notebook's copy of the repository in sync with GitHub each time you log in, and that sync can clash
with your own changes. A separate clone keeps your work safe.

The layout follows the phlex-examples README: the build directory sits next to the source directory.

```
~/phlex-work/
├── phlex-examples/          <- your clone (source)
├── build-phlex-examples/    <- CMake build directory
└── cmake-intro/             <- small CMake warm-up project
```

Each `!` cell runs in a new shell, so `cd` or `export` in one cell does not carry over to the next.
The next cell sets the paths as environment variables of the kernel, which every later cell inherits.
Run it again if you restart the kernel.

In [ ]:
import os
from pathlib import Path

WORK = Path.home() / "phlex-work"
SRC = WORK / "phlex-examples"
BUILD = WORK / "build-phlex-examples"
INTRO = WORK / "cmake-intro"
WORK.mkdir(exist_ok=True)

os.environ["WORK"] = str(WORK)
os.environ["SRC"] = str(SRC)
os.environ["BUILD"] = str(BUILD)
os.environ["INTRO"] = str(INTRO)

# phlex finds plugins (MODULE libraries) through PHLEX_PLUGIN_PATH and Python algorithms through PYTHONPATH.
for var, path in [("PHLEX_PLUGIN_PATH", BUILD), ("PYTHONPATH", SRC)]:
    entries = os.environ.get(var, "").split(":")
    if str(path) not in entries:
        os.environ[var] = ":".join([str(path)] + [e for e in entries if e])

for name in ["WORK", "SRC", "BUILD", "PHLEX_PLUGIN_PATH", "PYTHONPATH"]:
    print(f"{name:18} {os.environ[name]}")

# Section 1: Getting the code with git

git keeps the full history of a project. Each **commit** is a snapshot of every file, with an author,
a date and a message. A **branch** is a name that points at a commit and moves forward as you add
commits. A **clone** is your own complete copy of the repository, history included.

Clone phlex-examples with HTTPS. You don't need a GitHub account to clone a public repository.

In [ ]:
%%bash
cd "$WORK"
if [ -d phlex-examples ]; then
  echo "phlex-examples is already cloned, skipping."
else
  git clone https://github.com/Framework-R-D/phlex-examples.git
fi

## Look around

`git log` shows the history. `--oneline` prints one line per commit: a short **hash** (the commit's
ID) and the first line of its message.

In [ ]:
%%bash
cd "$SRC"
git log --oneline -8

`git branch -a` lists branches. The ones under `remotes/origin/` live on GitHub. Your clone has
one local branch, `main`.

phlex-examples keeps a branch for each Phlex release series: `phlex-v0.3`, `phlex-v0.4` and so on.
Phlex is still changing quickly, so the examples on `main` can need a newer Phlex than the one you
have. **Always use the branch that matches your Phlex version.** `phlex --version` in Section 0
told you which one.

In [ ]:
%%bash
cd "$SRC"
git branch -a

`git diff` compares two commits or branches. `--stat` summarises which files changed, and `-- <path>`
limits the comparison to some files (`':!migration'` leaves that directory out). This is a quick way
to see what you need to change in your own code when you move to a new Phlex release. Between v0.2 and
v0.4, for example, `product_query` was renamed `product_selector`.

In [ ]:
%%bash
cd "$SRC"
echo "=== Top-level files that changed between the v0.3 and v0.4 examples"
git diff --stat origin/phlex-v0.3 origin/phlex-v0.4 -- . ':!migration'
echo
echo "=== How the add module's registration changed between v0.2 and v0.4"
git diff origin/phlex-v0.2 origin/phlex-v0.4 -- add_and_verify.cpp

## Switch to the right branch and make your own

`git switch` moves your working files to another branch. Switch to the branch for your Phlex
version (change `phlex-v0.4` below if your version is different), then create your own branch with
`git switch -c`. Your commits go on your branch, and the release branch stays untouched.

git also needs to know who you are before you can commit. The `git config` lines below set your name
and email **for this repository only**. Replace them with your own.

In [ ]:
%%bash
cd "$SRC"
git switch phlex-v0.4
git switch -c my-workshop 2>/dev/null || git switch my-workshop

git config user.name  "Your Name"
git config user.email "you@example.org"

git status

`git status` is the command you will run most often. It tells you which branch you are on and which
files you have changed. Right now it should say `nothing to commit, working tree clean`.

# Section 2: What CMake does

C++ is compiled. Every `.cpp` file is compiled into an object file, and object files are linked into
libraries or executables. For a real project you also have to find dependencies (Phlex, Boost, TBB),
pass the right compiler flags, and rebuild only what changed. CMake handles all of that.

CMake works in two steps:

1. **Configure.** `cmake <source-dir>` reads `CMakeLists.txt`, finds the compiler and the
   dependencies, and writes build files (Makefiles by default) into the current directory. It stores
   what it found in `CMakeCache.txt`.
2. **Build.** `cmake --build .` (or `make`) runs the compiler and linker. It rebuilds only the files
   that changed since the last build.

You always build in a **separate build directory**, never inside the source directory. Your source
stays clean, and you can delete the build directory and start again at any time.

## A small warm-up project

Before phlex-examples, build a tiny project with one library and one program that uses it.

In [ ]:
from pathlib import Path

(INTRO / "src").mkdir(parents=True, exist_ok=True)

(INTRO / "src" / "greet.hpp").write_text('''\
#ifndef GREET_HPP
#define GREET_HPP

#include <string>

std::string greet(std::string const& name);

#endif
''')

(INTRO / "src" / "greet.cpp").write_text('''\
#include "greet.hpp"

std::string greet(std::string const& name) { return "Hello, " + name + "!"; }
''')

(INTRO / "src" / "hello.cpp").write_text('''\
#include "greet.hpp"

#include <iostream>

int main() { std::cout << greet("Bengaluru") << std::endl; }
''')

(INTRO / "src" / "CMakeLists.txt").write_text('''\
cmake_minimum_required(VERSION 3.31)
project(cmake_intro LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 20)

# A shared library, built from greet.cpp
add_library(greet SHARED greet.cpp)

# An executable that links against the library
add_executable(hello hello.cpp)
target_link_libraries(hello PRIVATE greet)
''')

print((INTRO / "src" / "CMakeLists.txt").read_text())

Each line of that `CMakeLists.txt` does one job:

| Line | What it does |
|---|---|
| `cmake_minimum_required` | Refuses to run with an older CMake than the project needs. |
| `project` | Names the project and says which languages it uses. |
| `set(CMAKE_CXX_STANDARD 20)` | Compiles with C++20. |
| `add_library(greet SHARED greet.cpp)` | Makes a **target** called `greet`: a shared library, `libgreet.so`. |
| `add_executable(hello hello.cpp)` | Makes a target called `hello`: a program. |
| `target_link_libraries(hello PRIVATE greet)` | `hello` uses `greet`, so link it in. |

Modern CMake is built around **targets**. You declare a target, then attach sources, libraries and
flags to it. Configure and build:

In [ ]:
%%bash
mkdir -p "$INTRO/build"
cd "$INTRO/build"
cmake ../src
echo "======== build ========"
cmake --build .
echo "======== run ========"
./hello

Now change only the library and rebuild. Watch the build output: CMake recompiles `greet.cpp` and
relinks, but it does not recompile `hello.cpp`.

In [ ]:
src = INTRO / "src" / "greet.cpp"
src.write_text(src.read_text().replace('"Hello, "', '"Namaskara, "'))
print(src.read_text())

In [ ]:
%%bash
cd "$INTRO/build"
cmake --build .
./hello

## SHARED and MODULE libraries

CMake has two kinds of dynamic library, and Phlex uses both:

- **SHARED** (`libgreet.so`): other code links against it when it is built. `hello` links to
  `greet`. In phlex-examples, the experiment code that does not know about Phlex (`my_add`,
  `my_geometry`) is SHARED.
- **MODULE** (`numbers_source.so`): nothing links against it. A program opens it while it runs
  (with `dlopen`). This is how Phlex loads **plugins**. When a configuration file says
  `cpp: 'add_and_verify'`, phlex searches `PHLEX_PLUGIN_PATH` for `add_and_verify.so` and loads it.

This split matters for DUNE. The physics algorithm lives in an ordinary library that you can test on
its own. A thin MODULE registers it with Phlex.

# Section 3: Reading the phlex-examples CMakeLists.txt

In [ ]:
!cat "$SRC/CMakeLists.txt"

New things compared with the warm-up project:

- `find_package(phlex REQUIRED)` finds an installed Phlex and makes its targets available, such as
  `phlex::module`. `Boost`, `TBB` and `spdlog` are found the same way. If configure fails here, the
  environment does not have that package.
- `add_compile_options(-Wall -Wpedantic -Werror)` turns warnings on and makes every warning an error.
  It applies to targets declared **after** it in the file.
- `add_library(my_add SHARED my_add.cpp)` is the framework-agnostic "experiment library".
- `add_library(add_and_verify MODULE add_and_verify.cpp)` is a Phlex plugin.
  `target_link_libraries(add_and_verify PRIVATE phlex::module my_add my_geometry)` gives it the
  Phlex registration API plus the two experiment libraries.
- `target_compile_options(add_and_verify PRIVATE -UNDEBUG)` keeps `assert()` active in this target.
  The examples use asserts to check their results.
- `add_subdirectory(migration)` reads `migration/CMakeLists.txt` too. That directory holds the
  GausHitFinder art-to-Phlex migration example.

`PRIVATE` means "this target needs it, but code that uses this target does not". You will see
`PUBLIC` and `INTERFACE` in larger projects.

# Section 4: Build phlex-examples

In [ ]:
%%bash
mkdir -p "$BUILD"
cd "$BUILD"
cmake "$SRC" -DCMAKE_BUILD_TYPE=Release

Configure should end with `Build files have been written to: …`. Now build. `-j 4` runs four
compile jobs at once. The hub is shared, so please don't use more.

In [ ]:
%%bash
cd "$BUILD"
cmake --build . -j 4

In [ ]:
%%bash
cd "$BUILD"
echo "=== Plugins and libraries in the build directory"
ls *.so

# Section 5: Run the examples

## A C++ workflow

A Phlex job is described by a Jsonnet configuration file with three parts:

- `driver` sets up the **data layers**. `generate_layers` makes a `job` with 10 `spill` children.
- `sources` provide data products. `numbers_source` provides `i` and `j` for every spill;
  `geometry_source` provides a geometry once for the whole job.
- `modules` hold the algorithms. `add_and_verify` adds `i` and `j` and checks the sum.
  `form_module` writes products to `output.root`.

In [ ]:
!cat "$SRC/test-cpp-workflow.jsonnet"

Current Phlex releases require a **stage** name for every job, given with `--stage` or a top-level
`stage:` field in the configuration. FORM records it with the products it writes. The README's
`phlex -c` lines leave it out, so add it as below.

Run from the build directory, as the README does:

In [ ]:
%%bash
cd "$BUILD"
phlex -c "$SRC/test-cpp-workflow.jsonnet" --stage examples -j 4
ls -l output.root

At the end you should see `Processed layers:` with `spill: 10`.

## An unfold

An **unfold** splits one product into many children on a new layer. Here each `spill` provides a
`std::vector<int>` with 3 entries, the unfold makes one `number` per entry, a transform squares each
one, and an observer checks the result. Read `split_square_verify.cpp`: the comment at the top
explains the splitter class.

In [ ]:
!sed -n 1,30p "$SRC/split_square_verify.cpp"

In [ ]:
%%bash
cd "$BUILD"
phlex -c "$SRC/test-cpp-unfold-workflow.jsonnet" --stage examples -j 4

You should see `number: 15` under `spill: 5`: 3 numbers × 5 spills.

## A Python algorithm

Phlex can also run Python functions. `subtract.py` registers an ordinary function as a transform.
phlex reads the type annotations (`i: int, j: int -> int`) to match it with C++ products. It finds
the module through `PYTHONPATH`, which Section 0 already set to include your clone.

In [ ]:
!sed -n 1,25p "$SRC/subtract.py"

In [ ]:
%%bash
cd "$BUILD"
phlex -c "$SRC/test-py-workflow.jsonnet" --stage examples -j 4

# Section 6: Exercise A: add a new algorithm

You will add a `multiply` transform that multiplies `i` by `j`, plus an observer that checks the
result. Since `j = -i`, the product is `-i²`, so it can never be positive.

This exercise uses all three tools: you write C++, register it with CMake, and record it with git.

## A.1: Write the plugin

Run the next cell to create `multiply_and_verify.cpp`, then open it in the editor and read it. Compare
it with `add_and_verify.cpp`. It follows the same pattern: a transform with two inputs and one output,
and an observer that reads the transform's output.

In [ ]:
%%writefile ~/phlex-work/phlex-examples/multiply_and_verify.cpp
#include "phlex/module.hpp"

#include <cassert>

namespace {
  int multiply(int i, int j) { return i * j; }
}

PHLEX_REGISTER_ALGORITHMS(m, config)
{
  using namespace phlex;

  auto const layer = config.get<std::string>("layer");

  m.transform("multiply", multiply, concurrency::unlimited)
    .input_family(product_selector{.creator = "input", .layer = layer, .suffix = "i"},
                  product_selector{.creator = "input", .layer = layer, .suffix = "j"})
    .output_product_suffixes("product");

  m.observe(
     "verify_product", [](int product) { assert(product <= 0); }, concurrency::unlimited)
    .input_family(product_selector{.creator = "multiply", .layer = layer});
}

Ask git what it thinks. The new file shows up as **untracked**: git sees it but is not recording it yet.

In [ ]:
!cd "$SRC" && git status

## A.2: Tell CMake about it

CMake does not know about the new file until you declare a target for it.
**Edit `~/phlex-work/phlex-examples/CMakeLists.txt` now** and add these lines at the end:

```cmake
# Exercise A: multiply example
add_library(multiply_and_verify MODULE multiply_and_verify.cpp)
target_link_libraries(multiply_and_verify PRIVATE phlex::module)
target_compile_options(multiply_and_verify PRIVATE -UNDEBUG)
```

Why each line: it is a plugin, so `MODULE`. It uses the Phlex registration API, so it links
`phlex::module`. It checks its result with `assert`, so it needs `-UNDEBUG` in a Release build.

If you are short of time, the solution cell below adds the lines for you. It does nothing if they are
already there.

In [ ]:
# Solution for A.2
cmake_file = SRC / "CMakeLists.txt"
text = cmake_file.read_text()
if "multiply_and_verify" not in text:
    text = text.rstrip("\n") + '''

# Exercise A: multiply example
add_library(multiply_and_verify MODULE multiply_and_verify.cpp)
target_link_libraries(multiply_and_verify PRIVATE phlex::module)
target_compile_options(multiply_and_verify PRIVATE -UNDEBUG)
'''
    cmake_file.write_text(text)
    print("Added the multiply_and_verify target.")
else:
    print("CMakeLists.txt already has multiply_and_verify.")

Rebuild. You don't need to run the configure step by hand: `cmake --build` sees that `CMakeLists.txt`
changed and reconfigures first. Only the new plugin gets compiled.

In [ ]:
%%bash
cd "$BUILD"
cmake --build . -j 4 && ls -l multiply_and_verify.so

If the build fails, read the **first** error message. With `-Werror`, an unused variable is enough to
stop the build.

## A.3: Add it to the workflow

**Edit `~/phlex-work/phlex-examples/test-cpp-workflow.jsonnet`** and add a module next to `add_cpp`:

```jsonnet
    multiply_cpp: {
      cpp: 'multiply_and_verify',
      layer: 'spill',
    },
```

The name `multiply_cpp` is your choice. `cpp: 'multiply_and_verify'` must match the CMake target
name, because that is the `.so` file phlex looks for.

In [ ]:
# Solution for A.3
jsonnet_file = SRC / "test-cpp-workflow.jsonnet"
text = jsonnet_file.read_text()
if "multiply_and_verify" not in text:
    anchor = "    // Write \"sum\" data product to output file"
    assert anchor in text, "test-cpp-workflow.jsonnet does not look like the original; edit it by hand"
    block = "    multiply_cpp: {\n      cpp: 'multiply_and_verify',\n      layer: 'spill',\n    },\n"
    jsonnet_file.write_text(text.replace(anchor, block + anchor))
    print("Added multiply_cpp.")
else:
    print("test-cpp-workflow.jsonnet already has multiply_cpp.")

In [ ]:
%%bash
cd "$BUILD"
phlex -c "$SRC/test-cpp-workflow.jsonnet" --stage examples -j 4

The job should finish as before. Your observer ran once per spill. If an assert had failed, phlex would
have stopped with `Assertion 'product <= 0' failed`.

**Try it:** add `'product'` to the `products` list of the `output` module, rerun, and the products of
your new transform are written to `output.root` too.

## A.4: Commit your work

See exactly what you changed, then commit it. `git diff` shows changes to files git already tracks;
the new `.cpp` file only appears in `git status` until you add it.

In [ ]:
%%bash
cd "$SRC"
git status --short
echo "=========================="
git diff

Committing is two steps. `git add` puts changes in the **staging area** (the next commit being
prepared). `git commit` records the staged snapshot with a message. Write messages that say what the
change does, so you can find it again later.

In [ ]:
%%bash
cd "$SRC"
git add multiply_and_verify.cpp CMakeLists.txt test-cpp-workflow.jsonnet
git commit -m "Add multiply transform and observer to the C++ workflow"
git log --oneline -3

Notice that you did not commit anything from `~/phlex-work/build-phlex-examples`. Build products
never go into git; they are rebuilt from the source.

# Section 7: Exercise B: git as a safety net

Everyone breaks working code sometimes. git makes it cheap to find what you changed and to undo it.

## B.1: Break something

**Edit `my_add.cpp`** and change `return i + j;` to `return i - j;`. (Or run the cell below, which
does it for you.)

In [ ]:
f = SRC / "my_add.cpp"
f.write_text(f.read_text().replace("return i + j;", "return i - j;"))
print(f.read_text())

Rebuild and look at what gets recompiled. `my_add` is a SHARED library, so only `libmy_add.so` is
rebuilt. `add_and_verify.so` loads the new version at run time without being recompiled.

In [ ]:
%%bash
cd "$BUILD"
cmake --build . -j 4

Now run the workflow. `j = -i`, so `i - j = 2i`, and the observer's `assert(sum == 0)` fails. **This
cell is supposed to fail.**

In [ ]:
%%bash
cd "$BUILD"
phlex -c "$SRC/test-cpp-workflow.jsonnet" --stage examples -j 4 2>&1 | tail -5
echo "phlex exit code: ${PIPESTATUS[0]}"

## B.2: Find the change and undo it

In a big project you might not remember what you edited. Ask git:

In [ ]:
!cd "$SRC" && git diff

`git restore <file>` throws away your uncommitted changes to that file and puts back the last
committed version. Use it with care: those changes are gone for good.

In [ ]:
%%bash
cd "$SRC"
git restore my_add.cpp
git status --short
cd "$BUILD"
cmake --build . -j 4
phlex -c "$SRC/test-cpp-workflow.jsonnet" --stage examples -j 4 | tail -4

Other ways back, for later:

| Situation | Command |
|---|---|
| Undo uncommitted edits to a file | `git restore <file>` |
| Unstage a file you `git add`-ed too early | `git restore --staged <file>` |
| Put work aside to try something else | `git stash`, and later `git stash pop` |
| Undo a commit you already made, by adding a new commit that reverses it | `git revert <hash>` |
| See a file as it was in an older commit | `git show <hash>:<file>` |

# Section 8: Exercise C: a Python observer

Add a Python algorithm that checks the `difference` produced by `subtract.py`. Since `j = -i`,
`i - j = 2i`, which is always even.

A Python observer is a function that returns `None`. phlex uses the function's name (`check`) as the
algorithm name, and the annotation `difference: int` tells phlex what type it expects.

In [ ]:
%%writefile ~/phlex-work/phlex-examples/check_difference.py
"""Check that the difference computed by subtract.py is even."""


def check(difference: int) -> None:
    """Fail the job if 'difference' is odd."""
    assert difference % 2 == 0, f"difference {difference} is odd"


def PHLEX_REGISTER_ALGORITHMS(m, config):
    """Register `check` as an observer of the configured input."""
    m.observe(check, input_family=config["input"])

**Edit `test-py-workflow.jsonnet`** and add this module after `subtract`:

```jsonnet
    check: {
      py: 'check_difference',
      input: [
        { creator: 'subtract', suffix: 'difference', layer: 'job' },
      ],
    },
```

`creator: 'subtract'` is the name of the Python function that made the product, and
`suffix: 'difference'` is its output, set by `output: ['difference']` in the `subtract` module.

In [ ]:
# Solution for Exercise C
jsonnet_file = SRC / "test-py-workflow.jsonnet"
text = jsonnet_file.read_text()
if "check_difference" not in text:
    anchor = "    // Python data products cannot yet be written to output files"
    assert anchor in text, "test-py-workflow.jsonnet does not look like the original; edit it by hand"
    block = (
        "    check: {\n"
        "      py: 'check_difference',\n"
        "      input: [\n"
        "        { creator: 'subtract', suffix: 'difference', layer: 'job' },\n"
        "      ],\n"
        "    },\n"
    )
    jsonnet_file.write_text(text.replace(anchor, block + anchor))
    print("Added check.")
else:
    print("test-py-workflow.jsonnet already has check.")

Python modules don't need to be built: there is nothing to add to `CMakeLists.txt`. Run the job, then
commit.

In [ ]:
%%bash
cd "$BUILD"
phlex -c "$SRC/test-py-workflow.jsonnet" --stage examples -j 4

In [ ]:
%%bash
cd "$SRC"
git add check_difference.py test-py-workflow.jsonnet
git commit -m "Add a Python observer that checks the subtract output"
git log --oneline -4

# Section 9 (optional): The GausHitFinder migration tests

`migration/gauss_hit_finder` is a real LArSoft module (`GausHitFinder`) converted to Phlex in four
designs. Its `CMakeLists.txt` defines **tests** with `add_test()`. Each test runs phlex and then
compares the hits it finds with the hits the original art module found (`art_hits_*.txt`).

CMake only registers tests when the top-level `CMakeLists.txt` calls `enable_testing()`. Check
whether yours does:

In [ ]:
!grep -n "enable_testing\|include(CTest)" "$SRC/CMakeLists.txt" || echo "No enable_testing() in the top-level CMakeLists.txt"

If it is missing, **add `enable_testing()`** to the top-level `CMakeLists.txt` on the line after
`find_package(phlex REQUIRED)`, then rebuild and run the tests with `ctest`. `-R` picks tests by name,
and `--output-on-failure` shows the output of any test that fails.

In [ ]:
# Solution: add enable_testing() after find_package(phlex REQUIRED)
cmake_file = SRC / "CMakeLists.txt"
text = cmake_file.read_text()
if "enable_testing()" not in text:
    anchor = "find_package(phlex REQUIRED)\n"
    assert anchor in text, "CMakeLists.txt does not look like the original; edit it by hand"
    cmake_file.write_text(text.replace(anchor, anchor + "enable_testing()\n", 1))
    print("Added enable_testing().")
else:
    print("CMakeLists.txt already calls enable_testing().")

In [ ]:
%%bash
cd "$BUILD"
cmake --build . -j 4
ctest -N | tail -3
ctest -R Design0 --output-on-failure

**Try it:** run `ctest` with no `-R` to run all four designs, and read
`migration/gauss_hit_finder/README.md` to see how the designs differ.

# Section 10: Sharing your work

Your commits are on the `my-workshop` branch in your clone. To see everything you did relative to the
release branch:

In [ ]:
%%bash
cd "$SRC"
git log --oneline phlex-v0.4..my-workshop
echo "==========================="
git diff --stat phlex-v0.4..my-workshop

To share work on GitHub you normally don't push to the original repository. The usual flow is:

1. **Fork** the repository on GitHub (the Fork button). This makes your own copy, such as
   `github.com/<you>/phlex-examples`.
2. Add your fork as a second **remote**:
   `git remote add myfork https://github.com/<you>/phlex-examples.git`
3. **Push** your branch to it: `git push myfork my-workshop`. HTTPS pushes need a GitHub personal
   access token as the password.
4. Open a **pull request** on GitHub from `my-workshop` in your fork to the branch you started from.
   Reviewers comment on it, you push more commits to the same branch, and a maintainer merges it.

This is the flow that DUNE code in Phlex will use, so it is worth practising on a toy change.

## Summary

| Tool | What you used |
|---|---|
| git | `clone`, `log`, `branch`, `switch`, `status`, `diff`, `add`, `commit`, `restore` |
| CMake | configure vs build, out-of-source builds, targets, `SHARED` vs `MODULE`, `find_package`, `target_link_libraries`, `ctest` |
| Phlex | Jsonnet configuration (`driver`, `sources`, `modules`), `--stage`, `PHLEX_PLUGIN_PATH`, transforms, observers, unfolds, Python algorithms |